In [0]:
-- Databricks notebook source
-- MAGIC %md
-- MAGIC # Gold Layer: Property Revenue & Amsterdam Postcode Profitability (Databricks SQL)
-- MAGIC - **Dataset Era:** 2019–2020 pre-pandemic Amsterdam market (`crawledAt`: Jul 2019 – Mar 2020)
-- MAGIC - **Occupancy Baseline:** 70% annualized short-term occupancy (2019 CBS / AirDNA Amsterdam benchmark) + 2019 Amsterdam 30-night vacation rental cap
-- MAGIC - **Property Normalization:** Aligns rental `area_sqm` with Airbnb `bedrooms`/`room_type` into comparable `size_tier` buckets
-- 1. Property-Level Revenue: rental (Amsterdam Only, Valid Postcodes, Size-Tiered)
CREATE OR REPLACE TABLE gold.rentals_fact
USING DELTA
AS
SELECT
  rental_id,
  city,
  pc4,
  pc6,
  property_type,
  CASE
    WHEN standardized_room_type = 'Private room' THEN '1_Room'
    WHEN property_type = 'Studio' OR (standardized_room_type = 'Entire home/apt' AND COALESCE(area_sqm, 40) <= 45) THEN '2_Compact_1Bed_or_Studio'
    WHEN standardized_room_type = 'Entire home/apt' AND area_sqm > 45 THEN '3_Large_2PlusBed_Apt'
    ELSE 'Other'
  END AS size_tier,
  area_sqm,
  base_monthly_rent,
  additional_monthly_costs,
  total_monthly_rent AS expected_monthly_revenue,
  ROUND(total_monthly_rent * 12, 2) AS expected_annual_revenue,
  ROUND(total_monthly_rent / NULLIF(area_sqm, 0), 2) AS monthly_rent_per_sqm,
  is_utilities_included,
  furnish,
  energy_label,
  latitude,
  longitude
FROM silver.rentals_clean
WHERE is_amsterdam = TRUE
  AND is_valid_zipcode = TRUE
  AND standardized_room_type IN ('Entire home/apt', 'Private room');

In [0]:
-- 2. Property-Level Revenue: Airbnb (Valid Postcodes, Size-Tiered)
DECLARE OR REPLACE VARIABLE assumed_occupancy_rate DOUBLE DEFAULT 0.70; -- Ref: 2019 CBS / AirDNA Amsterdam benchmark (~70-78%)
DECLARE OR REPLACE VARIABLE amsterdam_annual_night_cap INT DEFAULT 30;    -- Ref: Huisvestingsverordening Amsterdam (effective Jan 1, 2019)
CREATE OR REPLACE TABLE gold.airbnb_fact
USING DELTA
AS
SELECT
  airbnb_row_id,
  pc4,
  pc6,
  room_type,
  CASE
    WHEN room_type = 'Private room' THEN '1_Room'
    WHEN room_type = 'Entire home/apt' AND COALESCE(bedrooms, 1) <= 1 THEN '2_Compact_1Bed_or_Studio'
    WHEN room_type = 'Entire home/apt' AND bedrooms >= 2 THEN '3_Large_2PlusBed_Apt'
    ELSE 'Other'
  END AS size_tier,
  accommodates,
  bedrooms,
  price,
  review_scores_value,
  has_recommendation,
  -- Scenario A: Market Occupancy Benchmark (70% occupancy = ~255.5 nights/yr)
  ROUND(price * 30 * assumed_occupancy_rate, 2) AS market_occ_monthly_revenue,
  ROUND(price * 365 * assumed_occupancy_rate, 2) AS market_occ_annual_revenue,
  -- Scenario B: Amsterdam 2019 Statutory 30-Night Cap
  ROUND((price * amsterdam_annual_night_cap) / 12.0, 2) AS capped_30n_monthly_revenue,
  ROUND(price * amsterdam_annual_night_cap, 2) AS capped_30n_annual_revenue,
  latitude,
  longitude
FROM silver.airbnb_clean
WHERE is_valid_zipcode = TRUE
  AND room_type IN ('Entire home/apt', 'Private room');

In [0]:
-- 3. Primary Gold Mart: Amsterdam PC4 Postcode Profitability by Size Tier
CREATE OR REPLACE TABLE gold.postcode_pc4_profitability
USING DELTA
AS
WITH rental_pc4 AS (
  SELECT
    pc4,
    size_tier,
    COUNT(*) AS rental_listing_count,
    ROUND(AVG(expected_monthly_revenue), 2) AS rental_avg_monthly_rent,
    ROUND(PERCENTILE_APPROX(expected_monthly_revenue, 0.5), 2) AS rental_median_monthly_rent,
    ROUND(PERCENTILE_APPROX(expected_annual_revenue, 0.5), 2) AS rental_median_annual_revenue,
    ROUND(AVG(monthly_rent_per_sqm), 2) AS rental_avg_rent_per_sqm,
    ROUND(AVG(area_sqm), 1) AS rental_avg_area_sqm
  FROM gold.rentals_fact
  WHERE size_tier != 'Other'
  GROUP BY pc4, size_tier
),
airbnb_pc4 AS (
  SELECT
    pc4,
    size_tier,
    COUNT(*) AS airbnb_listing_count,
    SUM(CASE WHEN has_recommendation THEN 1 ELSE 0 END) AS airbnb_recommended_count,
    ROUND(AVG(price), 2) AS airbnb_avg_price,
    ROUND(PERCENTILE_APPROX(price, 0.5), 2) AS airbnb_median_price,
    ROUND(PERCENTILE_APPROX(market_occ_annual_revenue, 0.5), 2) AS airbnb_median_market_annual_rev,
    ROUND(PERCENTILE_APPROX(capped_30n_annual_revenue, 0.5), 2) AS airbnb_median_capped_30n_annual_rev,
    ROUND(AVG(review_scores_value), 1) AS airbnb_avg_review_score
  FROM gold.airbnb_fact
  WHERE size_tier != 'Other'
  GROUP BY pc4, size_tier
),
combined AS (
  SELECT
    COALESCE(k.pc4, a.pc4) AS pc4,
    COALESCE(k.size_tier, a.size_tier) AS size_tier,
    COALESCE(k.rental_listing_count, 0) AS rental_listing_count,
    COALESCE(a.airbnb_listing_count, 0) AS airbnb_listing_count,
    COALESCE(a.airbnb_recommended_count, 0) AS airbnb_recommended_count,
    k.rental_avg_monthly_rent,
    k.rental_median_monthly_rent,
    k.rental_median_annual_revenue,
    k.rental_avg_rent_per_sqm,
    k.rental_avg_area_sqm,
    a.airbnb_avg_price,
    a.airbnb_median_price,
    a.airbnb_median_market_annual_rev,
    a.airbnb_median_capped_30n_annual_rev,
    a.airbnb_avg_review_score,
    ROUND(a.airbnb_median_market_annual_rev - k.rental_median_annual_revenue, 2) AS market_occ_annual_diff_eur,
    ROUND(a.airbnb_median_capped_30n_annual_rev - k.rental_median_annual_revenue, 2) AS capped_30n_annual_diff_eur,
    ROUND(k.rental_median_annual_revenue / NULLIF(a.airbnb_median_price, 0), 1) AS airbnb_breakeven_nights_per_year
  FROM rental_pc4 k
  FULL OUTER JOIN airbnb_pc4 a
    ON k.pc4 = a.pc4
   AND k.size_tier = a.size_tier
)
SELECT
  *,
  CASE
    WHEN rental_listing_count = 0 OR airbnb_listing_count = 0 THEN 'Insufficient Pair Data'
    WHEN capped_30n_annual_diff_eur > 0 THEN 'Airbnb More Profitable (Even Under 30-Night Cap)'
    WHEN market_occ_annual_diff_eur > 0 THEN 'rental Safer (Airbnb Wins Only If Uncapped > ' || CAST(ROUND(airbnb_breakeven_nights_per_year, 0) AS STRING) || ' Nights)'
    ELSE 'rental More Profitable'
  END AS investment_verdict,
  CASE
    WHEN rental_listing_count >= 5 AND airbnb_listing_count >= 5 THEN 'HIGH'
    WHEN rental_listing_count >= 2 AND airbnb_listing_count >= 2 THEN 'MEDIUM'
    ELSE 'LOW'
  END AS data_confidence_tier
FROM combined;

In [0]:
-- 4. Secondary Gold View: Exact PC6 Street-Level Overlap (Where Both Sources Have Valid PC6)
CREATE OR REPLACE VIEW gold.postcode_pc6_exact_overlap AS
SELECT
  k.pc6,
  k.pc4,
  k.size_tier,
  COUNT(DISTINCT k.rental_id) AS rental_count,
  COUNT(DISTINCT a.airbnb_row_id) AS airbnb_count,
  ROUND(AVG(k.expected_annual_revenue), 2) AS rental_avg_annual_rev,
  ROUND(AVG(a.market_occ_annual_revenue), 2) AS airbnb_avg_market_annual_rev,
  ROUND(AVG(a.market_occ_annual_revenue) - AVG(k.expected_annual_revenue), 2) AS avg_annual_diff_eur
FROM gold.rentals_fact k
INNER JOIN gold.airbnb_fact a
  ON k.pc6 = a.pc6
 AND k.size_tier = a.size_tier
WHERE k.pc6 IS NOT NULL
  AND a.pc6 IS NOT NULL
  AND k.size_tier != 'Other'
GROUP BY k.pc6, k.pc4, k.size_tier;

In [0]:
-- 5. Preview Top High-Confidence Postcodes
SELECT
  pc4,
  size_tier,
  rental_listing_count,
  airbnb_listing_count,
  rental_median_monthly_rent,
  airbnb_median_price,
  rental_median_annual_revenue,
  airbnb_median_market_annual_rev,
  airbnb_breakeven_nights_per_year,
  investment_verdict
FROM gold.postcode_pc4_profitability
WHERE data_confidence_tier IN ('HIGH', 'MEDIUM')
ORDER BY rental_median_annual_revenue DESC
LIMIT 20;